# 学習なしで好きなラベルの固有表現を抽出する (GLiNER)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mamorlis/nlpcookbook/blob/main/notebooks/06_parsing_ner/gliner.ipynb)

NLP クックブック 06-08 ／ 構文解析と固有表現抽出

「病名」「薬品名」「役職」のような独自のラベルで固有表現抽出をしたいが、学習データがない。

In [1]:
!pip install -q gliner fugashi unidic-lite

In [2]:
from gliner import GLiNER

model = GLiNER.from_pretrained("urchade/gliner_multi-v2.1")

text = "Dr. Sarah Chen, CTO of Anthropic, presented the Claude model at NeurIPS 2024 in Vancouver."
for e in model.predict_entities(text, ["person", "job title", "company", "product", "conference", "city"], threshold=0.4):
    print(f"{e['text']:14}{e['label']:12}{e['score']:.2f}")

Dr. Sarah Chenperson      0.96
CTO           job title   0.73
Anthropic     company     0.96
Claude model  product     0.96
NeurIPS 2024  conference  0.98
Vancouver     city        0.98


In [3]:
# 日本語は空白で区切られていないので、分かち書きしてから渡す
from fugashi import Tagger

wakati = Tagger("-Owakati")
text = "田中医師は、インフルエンザと診断された患者にタミフルを5日分処方し、慶應義塾大学病院への紹介状を書いた。"
labels = ["人名", "病名", "薬品名", "医療機関", "期間"]
for e in model.predict_entities(wakati.parse(text).strip(), labels, threshold=0.5):
    print(f"{e['text'].replace(' ', ''):\u3000<10}{e['label']:\u3000<6}{e['score']:.2f}")

田中医師　　　　　　人名　　　　0.95
インフルエンザ　　　病名　　　　0.97
タミフル　　　　　　薬品名　　　0.70
5日分　　　　　　　期間　　　　0.83
慶應義塾大学病院　　医療機関　　0.94


- ラベル名を自然言語で与えるだけで抽出できる小型のエンコーダモデル。CPU でも動く。
- 内部で空白区切りの単語を単位にしているため、日本語や中国語は分かち書きが必要 (そのまま渡すと文全体が 1 語扱いになる)。
- ラベル名の言い回しで結果が変わる。`threshold` を下げると再現率が上がる。
- LLM に抽出させる方法と比べて、高速で、原文中の位置 (`start`/`end`) が確実に得られるのが利点。